# Surface Geometry (Normal Maps) — HummingBird2

Estimates per-pixel surface normals using **StableNormal (YOSO)** from HuggingFace, then computes per-body-part roughness statistics using the SAM 2 masks.

**Runtime:** Runtime > Change runtime type > **T4 GPU**

**Before running, make sure these files are in `MyDrive/Iridescence Lab/HummingBird2/`:**
1. The video file (`HummingBird2_1.mov`)
2. `masks.npz` (from the Segment tab)

**After running:** Download `normal_data.csv` and upload it in the Surface Geometry tab.

In [ ]:
# ── Install StableNormal and dependencies ──────────────────────────────────
!pip install -q git+https://github.com/Stable-X/StableNormal.git
!pip install -q diffusers transformers accelerate
print('Install complete.')

In [ ]:
# ── Imports ─────────────────────────────────────────────────────────────────
import os, cv2
import numpy as np
import pandas as pd
import torch
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}  |  device: {device}')

In [ ]:
# ── Mount Drive ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
SPECIES       = 'HummingBird2'
DRIVE_FOLDER  = '/content/drive/MyDrive/Iridescence Lab/HummingBird2'
VIDEO_PATH    = DRIVE_FOLDER + '/HummingBird2_1.mov'
MASKS_PATH    = DRIVE_FOLDER + '/masks.npz'
NORMALS_OUT   = DRIVE_FOLDER + '/normal_data.csv'
FRAME_STRIDE  = 1  # process every Nth frame (1 = all)

assert os.path.exists(VIDEO_PATH),  'Video not found: '  + VIDEO_PATH
assert os.path.exists(MASKS_PATH),  'masks.npz not found — run segmentation first'
print('Ready:', SPECIES)

In [ ]:
# ── Extract frames from video ───────────────────────────────────────────────
FRAMES_DIR = '/content/frames'
os.makedirs(FRAMES_DIR, exist_ok=True)

cap    = cv2.VideoCapture(VIDEO_PATH)
fps    = cap.get(cv2.CAP_PROP_FPS)
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'{width}x{height} @ {fps:.1f} FPS — {total} frames')

saved = 0
while True:
    ret, frame = cap.read()
    if not ret: break
    cv2.imwrite(f'{FRAMES_DIR}/{saved:05d}.jpg', frame,
                [cv2.IMWRITE_JPEG_QUALITY, 95])
    saved += 1
cap.release()
print(f'Extracted {saved} frames.')

In [ ]:
# ── Load masks.npz ───────────────────────────────────────────────────────────
masks_data = np.load(MASKS_PATH, allow_pickle=False)

video_segments = {}
parts = set()
for key in masks_data.files:
    if key.startswith('_'):
        continue
    fidx_str, part = key.split('_', 1)
    video_segments.setdefault(int(fidx_str), {})[part] = \
        masks_data[key].astype(bool)
    parts.add(part)

parts = sorted(parts)
print(f'Parts: {parts}')
print(f'Frames with masks: {len(video_segments)}')

In [ ]:
# ── Load StableNormal (YOSO — fast, single-pass) ───────────────────────────
from stablenormal.pipeline_yoso_normal import YOSONormalsPipeline

pipe = YOSONormalsPipeline.from_pretrained(
    'Stable-X/yoso-normal-v0-3',
    trust_remote_code=True,
    variant='fp16',
    torch_dtype=torch.float16,
).to(device)
pipe.set_progress_bar_config(disable=True)
print('StableNormal (YOSO) loaded.')

In [ ]:
# ── Estimate normals and compute per-part statistics ────────────────────────
#
# For each frame: run YOSO → (H,W,3) normal map in [-1,1]
# Apply each body-part mask → compute mean normal + angular roughness
#   roughness = std of per-pixel angle deviations from the mean normal
#   A flat mirror surface → roughness ≈ 0°
#   A rough matte surface → roughness > 20°

frame_idxs = sorted(video_segments.keys())
if FRAME_STRIDE > 1:
    frame_idxs = frame_idxs[::FRAME_STRIDE]
    print(f'Stride={FRAME_STRIDE}: processing {len(frame_idxs)} of '
          f'{len(video_segments)} frames')
else:
    print(f'Processing all {len(frame_idxs)} frames')

rows = []

for i, fidx in enumerate(frame_idxs):
    frame_path = f'{FRAMES_DIR}/{fidx:05d}.jpg'
    if not os.path.exists(frame_path):
        continue

    img_pil = Image.open(frame_path).convert('RGB')
    orig_w, orig_h = img_pil.size

    with torch.no_grad():
        normal_img = pipe(img_pil)   # PIL Image, RGB = packed XYZ

    # Decode: RGB 0-255 → normal XYZ in [-1, 1]
    normal_np = np.array(normal_img.resize((orig_w, orig_h),
                                           Image.BILINEAR)).astype(np.float32)
    normal_np = normal_np / 127.5 - 1.0   # (H, W, 3)

    # Renormalise each pixel to unit length
    norms = np.linalg.norm(normal_np, axis=2, keepdims=True)
    normal_np /= np.maximum(norms, 1e-8)

    segs = video_segments.get(fidx, {})
    row  = {'frame': fidx}

    for part in parts:
        mask = segs.get(part)
        if mask is None or not mask.any():
            for s in ['nx', 'ny', 'nz', 'roughness']:
                row[f'{part}_{s}'] = np.nan
            continue

        pix = normal_np[mask]           # (N, 3)

        mean_n = pix.mean(axis=0)
        mean_n /= max(np.linalg.norm(mean_n), 1e-8)

        # Angular deviation from mean (roughness in degrees)
        dots   = np.clip((pix * mean_n).sum(axis=1), -1.0, 1.0)
        angles = np.degrees(np.arccos(dots))

        row[f'{part}_nx']        = round(float(mean_n[0]), 5)
        row[f'{part}_ny']        = round(float(mean_n[1]), 5)
        row[f'{part}_nz']        = round(float(mean_n[2]), 5)
        row[f'{part}_roughness'] = round(float(np.std(angles)), 3)

    rows.append(row)

    if (i + 1) % 25 == 0 or i == len(frame_idxs) - 1:
        print(f'  {i+1}/{len(frame_idxs)} frames done')

print(f'\nProcessed {len(rows)} frames.')

In [ ]:
# ── Save normal_data.csv to Google Drive ─────────────────────────────────────
df = pd.DataFrame(rows).set_index('frame')
df['method'] = 'stablenormal'
df.to_csv(NORMALS_OUT)
print(f'Saved {len(df)} rows  →  {NORMALS_OUT}')

In [ ]:
# ── Save normal map images to Google Drive ───────────────────────────────────
# Save one coloured normal-map PNG per sampled frame.
# RGB encodes XYZ orientation:
#   Red   = surface tilting left/right
#   Green = surface tilting up/down
#   Blue  = surface facing the camera

N_SAMPLE_MAPS = 20   # how many frames to save as images
MAPS_OUT      = DRIVE_FOLDER + '/normal_maps'
os.makedirs(MAPS_OUT, exist_ok=True)

step       = max(1, len(frame_idxs) // N_SAMPLE_MAPS)
sample_idx = frame_idxs[::step][:N_SAMPLE_MAPS]

print(f'Saving {len(sample_idx)} normal-map images...')

for fidx in sample_idx:
    frame_path = f'{FRAMES_DIR}/{fidx:05d}.jpg'
    if not os.path.exists(frame_path):
        continue

    img_pil = Image.open(frame_path).convert('RGB')
    orig_w, orig_h = img_pil.size

    with torch.no_grad():
        normal_img = pipe(img_pil)

    # Resize to original frame resolution and save
    if hasattr(normal_img, 'resize'):
        normal_img = normal_img.resize((orig_w, orig_h), Image.BILINEAR)
    out_path = f'{MAPS_OUT}/{fidx:05d}_normal.png'
    normal_img.save(out_path)

print(f'Saved {len(sample_idx)} images to {MAPS_OUT}')

# Zip for easy download
import shutil
ZIP_OUT = DRIVE_FOLDER + '/normal_maps'
shutil.make_archive(ZIP_OUT, 'zip', MAPS_OUT)
zip_mb = os.path.getsize(ZIP_OUT + '.zip') / 1e6
print(f'Zipped  →  {ZIP_OUT}.zip  ({zip_mb:.1f} MB)')
print()
print('Download both files from Google Drive and upload in the Surface Geometry tab:')
print('  · normal_data.csv')
print('  · normal_maps.zip')